# World3 (Limits to Growth)

*System Dynamics Studio, model library: Complex behavior*

The World3 model was created for the Club of Rome to study the long-term consequences of growth in a finite world. It demonstrates "overshoot and collapse": as population and industry grow, they consume non-renewable resources and generate pollution. Eventually, resource scarcity and environmental degradation cause a decline in industrial output, leading to a collapse in population. It remains a foundational model for sustainability science.

**Concepts:** Overshoot and collapse, Interconnected sectors, Policy experiments

**Expected behavior:** Growth in capital and population, followed by decline as resources deplete and pollution rises.

**Limitations:** A four-stock teaching sketch inspired by the structure of World3 (Meadows et al., 1972). It is not the calibrated World3 model and does not reproduce its published scenarios.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{Population}}{dt} &= \text{Births} - \text{Deaths} \\
\frac{d\,\text{Industrial\_Capital}}{dt} &= \text{Investment} - \text{Depreciation} \\
\frac{d\,\text{Resources}}{dt} &= - \text{Resource\_Usage} \\
\frac{d\,\text{Pollution}}{dt} &= \text{Pollution\_Gen} - \text{Pollution\_Abs} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `Population` | 100 | — |
| `Industrial_Capital` | 100 | — |
| `Resources` | 100 | — |
| `Pollution` | 10 | — |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{Ind\_Output} &= \text{Industrial\_Capital}\cdot\left(\frac{\text{Resources}}{100}\right) \\
\text{Ind\_Cap\_Per\_Capita} &= \frac{\text{Industrial\_Capital}}{\text{Population}} \\
\text{Births} &= \text{Population}\cdot0.04 \\
\text{Deaths} &= \text{Population}\cdot\left(0.01+\text{Pollution}\cdot1\cdot10^{-4}+\frac{0.05}{\left(\text{Ind\_Cap\_Per\_Capita}+0.1\right)}\right) \\
\text{Investment} &= \text{Ind\_Output}\cdot0.1 \\
\text{Depreciation} &= \text{Industrial\_Capital}\cdot0.05 \\
\text{Resource\_Usage} &= \text{Ind\_Output}\cdot0.01 \\
\text{Pollution\_Gen} &= \text{Ind\_Output}\cdot0.05 \\
\text{Pollution\_Abs} &= \text{Pollution}\cdot0.1 \\
\end{aligned}
$$

This model has no separate parameters: its constants are written directly into the equations above.

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 200 (time unit not set), output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
}

# 2. Initial State (Stocks)
y0 = [
    100,  # Population
    100,  # Industrial_Capital
    100,  # Resources
    10  # Pollution
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    Population, Industrial_Capital, Resources, Pollution = y

    # Auxiliaries & Flows (dependency order)
    Ind_Output = (Industrial_Capital * ((Resources / 100)))
    Ind_Cap_Per_Capita = (Industrial_Capital / Population)
    Births = (Population * 0.04)
    Deaths = (Population * (((0.01 + (Pollution * 0.0001)) + (0.05 / ((Ind_Cap_Per_Capita + 0.1))))))
    Investment = (Ind_Output * 0.1)
    Depreciation = (Industrial_Capital * 0.05)
    Resource_Usage = (Ind_Output * 0.01)
    Pollution_Gen = (Ind_Output * 0.05)
    Pollution_Abs = (Pollution * 0.1)

    # Derivatives (Change in Stocks)
    dPopulation_dt = Births - Deaths
    dIndustrial_Capital_dt = Investment - Depreciation
    dResources_dt = - Resource_Usage
    dPollution_dt = Pollution_Gen - Pollution_Abs

    return [dPopulation_dt, dIndustrial_Capital_dt, dResources_dt, dPollution_dt]

## Simulation

The model runs from t = 0 to t = 200 time units. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 200, 2001)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
Population = sol[:, 0]
Industrial_Capital = sol[:, 1]
Resources = sol[:, 2]
Pollution = sol[:, 3]
Ind_Output = (Industrial_Capital * ((Resources / 100))) + np.zeros_like(t)
Ind_Cap_Per_Capita = (Industrial_Capital / Population) + np.zeros_like(t)
Births = (Population * 0.04) + np.zeros_like(t)
Deaths = (Population * (((0.01 + (Pollution * 0.0001)) + (0.05 / ((Ind_Cap_Per_Capita + 0.1)))))) + np.zeros_like(t)
Investment = (Ind_Output * 0.1) + np.zeros_like(t)
Depreciation = (Industrial_Capital * 0.05) + np.zeros_like(t)
Resource_Usage = (Ind_Output * 0.01) + np.zeros_like(t)
Pollution_Gen = (Ind_Output * 0.05) + np.zeros_like(t)
Pollution_Abs = (Pollution * 0.1) + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, Population, label='Population', linewidth=2)
plt.plot(t, Industrial_Capital, label='Industrial_Capital', linewidth=2)
plt.plot(t, Resources, label='Resources', linewidth=2)
plt.plot(t, Pollution, label='Pollution', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Doubling the initial resource base: does the collapse disappear, or is it only postponed?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['Population', 'Industrial_Capital', 'Resources', 'Pollution']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={}, initial_changes={'Resources': 200})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='Resources(0) = 200', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- Which stock peaks first? Which feedback explains the turn?
- Change the pollution absorption fraction. Does the collapse disappear, or only shift?